In [1]:
import torch

def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

get_device()

device(type='cuda')

# Load Dataset

In [2]:
import pandas as pd
from datasets import Dataset
try: # Colab
    df = pd.read_csv("../data/train.csv")
    test_df = pd.read_csv("../data/test.csv")
except: # Local
    df = pd.read_csv("train.csv")
    test_df = pd.read_csv("test.csv")

In [30]:
dataset = Dataset.from_pandas(df)
dataset_test = Dataset.from_pandas(test_df)

dataset, dataset_test

(Dataset({
     features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer'],
     num_rows: 2000
 }),
 Dataset({
     features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E'],
     num_rows: 500
 }))

# Building a neural network to work with this

## Build a Byte Pair Encoding corpus on the dataset

In [4]:
dataset

Dataset({
    features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer'],
    num_rows: 2000
})

In [5]:
from tokenizers import Tokenizer, models, pre_tokenizers, decoders, trainers
from transformers import PreTrainedTokenizerFast

# generator function for memory efficiency
def batched_generator(dataset, batch_size=1000):
    for i in range(0, len(dataset), batch_size):
        batch = dataset[i : i + batch_size]
        
        combined_texts = []
        for p, a, b, c, d, e in zip(
            batch["prompt"], batch["A"], batch["B"], batch["C"], batch["D"], batch["E"]
        ):
            text = f"{p} {a} {b} {c} {d} {e}"
            combined_texts.append(text)
            
        yield combined_texts

# tokenizer setup
tokenizer_model = Tokenizer(models.BPE(unk_token="[UNK]"))
tokenizer_model.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer_model.decoder = decoders.ByteLevel()


trainer = trainers.BpeTrainer(
    vocab_size=30_000,
    special_tokens=["[PAD]", "[UNK]", "[BOS]", "[EOS]"],
)

tokenizer_model.train_from_iterator(batched_generator(dataset), trainer=trainer)

tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer_model,
    pad_token="[PAD]",
    unk_token="[UNK]",
    bos_token="[BOS]",
    eos_token="[EOS]",
    tensors="pt"
)

tokenizer

TokenizersBackend(name_or_path='', vocab_size=6347, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'bos_token': '[BOS]', 'eos_token': '[EOS]', 'unk_token': '[UNK]', 'pad_token': '[PAD]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[BOS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[EOS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

---
testing the tokenizer

In [6]:
text = ["he is a good boy", "she is a good girl"]
encoded_tokens = tokenizer.encode(text)
decoded_text = tokenizer.decode(encoded_tokens)

# another method
# tokenizer(text[0], text[1], padding=True, truncation=True, return_tensors="pt")
encoded_tokens


[[119, 144, 118, 1863, 1070, 80], [74, 119, 144, 118, 1863, 174, 236, 67]]

---

## Creating preprocessor

In [7]:
(set(dataset['answer'][:]))


{'A', 'B', 'C', 'D', 'E'}

In [ ]:
def preprocess_dataset(dataset):
    texts = [
        f"Question: {p} Choice A: {a} Choice B: {b} Choice C: {c} Choice D: {d} Choice E: {e}"
        for p, a, b, c, d, e in zip(
            dataset['prompt'], dataset['A'], dataset['B'], dataset['C'], dataset['D'], dataset['E']
        )
    ]
    
    output = tokenizer(
        texts, 
        padding=True, 
        truncation=True, 
        max_length=256
    )
    
    # Target label integer (0 to 4)
    if 'answer' in dataset:
        label2id = {label: i for i, label in enumerate(sorted(set(dataset['answer'])))}
        output['labels'] = [label2id[label] for label in dataset['answer']]
        
    return output


test_encoded_dataset = preprocess_dataset(dataset[:100])
test_encoded_dataset

{'input_ids': [[46, 76, 450, 134, 25, 389, 581, 122, 765, 708, 475, 25, 244, 144, 2360, 2385, 300, 1784, 196, 122, 745, 339, 402, 156, 3148, 1460, 28, 657, 122, 659, 655, 13, 372, 432, 1297, 231, 25, 2360, 2385, 3147, 172, 4070, 874, 1076, 118, 402, 4451, 172, 144, 2333, 156, 895, 411, 470, 118, 972, 3980, 198, 4047, 13, 168, 745, 158, 122, 2394, 1879, 5241, 271, 228, 118, 5180, 5187, 11, 156, 122, 745, 158, 122, 2755, 1879, 1513, 118, 1772, 172, 702, 5200, 5095, 488, 300, 4103, 402, 13, 372, 432, 1297, 592, 25, 2360, 2385, 3147, 172, 4070, 597, 411, 874, 3410, 402, 11, 541, 172, 1179, 213, 402, 13, 168, 745, 158, 122, 2394, 144, 118, 2113, 5185, 130, 3993, 1895, 11, 156, 122, 745, 158, 122, 2755, 1879, 5242, 118, 1834, 5127, 11, 5216, 11, 198, 5154, 13, 372, 432, 1297, 372, 25, 2360, 2385, 895, 411, 5157, 141, 122, 1460, 130, 402, 198, 172, 271, 530, 779, 333, 196, 3148, 5244, 13, 168, 745, 158, 122, 2394, 156, 122, 2755, 144, 5249, 11, 156, 3148, 1460, 144, 2678, 477, 196, 122, 2113,

In [9]:
print(' '.join([str(int(i)) for i in test_encoded_dataset.labels[:10]])) 
print(' '.join(dataset[:100]['answer'][:10]))

# works

1 0 2 1 0 2 4 0 0 0
B A C B A C E A A A


In [10]:
import itertools
y = [dataset[i][option] for option in ['A', 'B', 'C', 'D', 'E'] for i in range(50)]

# shape of (2000, 5)
# if flatten becomes 10k
# list(itertools.chain.from_iterable(y))
len(y) # already flattened

250

---
## Building the model

In [11]:
vocab_size = tokenizer.vocab_size
embed_dim = 32
num_classes = 5
pad_id = tokenizer.pad_token_id

print(vocab_size, embed_dim, num_classes, pad_id)

6347 32 5 0


In [ ]:
from torch import nn
class MLPClassifierHF(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=256, num_classes=5, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
                
        # 4 layer mlp model
        self.net = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim * 2),
            nn.BatchNorm1d(hidden_dim * 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(hidden_dim // 2, num_classes)
        )
        self.num_classes = num_classes

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        embs = self.embedding(input_ids)                    # (B, T, D)
        mask = attention_mask.unsqueeze(-1).float()          # (B, T, 1)
        sum_embs = (embs * mask).sum(dim=1)                 # (B, D)
        avg_embs = sum_embs / mask.sum(dim=1).clamp(min=1)  # (B, D)
        logits = self.net(avg_embs)                         # (B, num_classes)

        loss = None
        if labels is not None:
            loss = nn.functional.cross_entropy(logits, labels)

        return {"loss": loss, "logits": logits} if loss is not None else {"logits": logits}


model = MLPClassifierHF(
    vocab_size=vocab_size,
    embed_dim=embed_dim,
    num_classes=num_classes,
    hidden_dim=64,
    pad_idx=pad_id,
)

model

MLPClassifierHF(
  (embedding): Embedding(6347, 32, padding_idx=0)
  (net): Sequential(
    (0): Linear(in_features=32, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): GELU(approximate='none')
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=64, out_features=32, bias=True)
    (9): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): GELU(approximate='none')
    (11): Dropout(p=0.2, inplace=False)
    (12): Linear(in_features=32, out_features=5, bias=True)
  )
)

---
# Evaluating the model

In [ ]:
import numpy as np
from transformers import Trainer, TrainingArguments

encoded_dataset = dataset.map(
    preprocess_dataset,
    batched=True,
    batch_size=100,
    remove_columns=dataset.column_names
)

split_dataset = encoded_dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split_dataset['train']
test_dataset = split_dataset['test']

def map3_metric(y_true, y_pred_top3):
    scores = []
    for true, top_k in zip(y_true, y_pred_top3):
        if true in top_k:
            rank = np.where(top_k == true)[0][0] + 1
            scores.append(1.0 / rank)
        else:
            scores.append(0.0)
    return float(np.mean(scores))

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    
    top3_preds = np.argsort(logits, axis=1)[:, ::-1][:, :3]
    
    map3_score = map3_metric(labels, top3_preds)
    return {"map@3": map3_score}

val_args = TrainingArguments(
    output_dir="./eval_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=1e-3,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
) # documentation copy paste :)

val_model = MLPClassifierHF(
    vocab_size=vocab_size,
    embed_dim=embed_dim,
    num_classes=num_classes,
    hidden_dim=64,
    pad_idx=pad_id,
)

val_trainer = Trainer(
    model=val_model,
    args=val_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

val_trainer.train()
eval_metrics = val_trainer.evaluate()
eval_metrics

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Map@3
1,1.556717,1.483763,0.638333
2,1.390464,1.286831,0.788750
3,1.326702,1.244028,0.809167


Training Loss,Validation Loss,Epoch,Map@3
1.326702,1.244028,3,0.809167



=== Validation Metrics (20% Test Set) ===
MAP@3: 0.8092
Loss:     1.2440



# Final train on full data

In [27]:
training_args = TrainingArguments(
    output_dir="./full_train_results",
    save_strategy="epoch",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=1e-3,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset,
    processing_class=tokenizer,
)

trainer.train()

Step,Training Loss
50,1.528519
100,1.362253
150,1.270823


TrainOutput(global_step=189, training_loss=1.3520277517813224, metrics={'train_runtime': 3.729, 'train_samples_per_second': 1609.028, 'train_steps_per_second': 50.684, 'total_flos': 0.0, 'train_loss': 1.3520277517813224, 'epoch': 3.0})

---
# creating submission file

In [29]:
test_dataset

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 400
})

In [31]:
encoded_test = dataset_test.map(
    preprocess_dataset,
    batched=True,
    batch_size=100,
    remove_columns=dataset_test.column_names
)
encoded_test

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 500
})

In [44]:
preds = trainer.predict(encoded_test).predictions
preds = torch.tensor(preds)
preds

tensor([[ 0.9348,  0.2900, -0.7421, -0.4741, -0.6770],
        [ 0.7369,  0.1075, -0.8415, -0.1227, -0.8445],
        [ 0.1534,  0.5507, -0.3934, -0.7417, -0.2982],
        ...,
        [ 0.5534,  0.8839, -0.7351, -0.8999, -1.1800],
        [-0.1496, -0.2156, -0.1408, -0.6577,  0.6339],
        [-0.5313, -0.1304,  0.7877, -0.3737,  0.6550]])

In [56]:
# select top 3 predictions for each example
topk = torch.topk(preds, k=3, dim=1).indices
labels = {
    0: "A",
    1: "B",
    2: "C",
    3: "D",
    4: "E"
}
predictions = [
    f"{labels[item[0]]} {labels[item[1]]} {labels[item[2]]}" for item in topk.tolist()
]
predictions[:5]

['A B D', 'A B D', 'B A E', 'E B A', 'C A D']

In [59]:
submission_df = pd.DataFrame({
    "ID": test_df["id"],
    "Prediction": predictions
})
submission_df.to_csv("submission.csv", index=False)